# Train / Validation / Test Split (time-group level)

Stratified 70/15/15 over **time groups**, not individual images: a burst of frames shot in the same second shows the same fish eye, so splitting per image leaks near-duplicates across subsets. Drawn once with a split seed separate from the training seeds, verified, and written to `02_Manifests/split_manifest.csv`, which every later notebook reads back.

The committed `split_manifest.csv` is the authoritative split. Re-running this notebook reproduces it exactly; it does not need to be run again before training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

import sys
sys.path.append('/content/repo/04_Src')

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'

import os
os.makedirs('/content/data', exist_ok=True)
!unzip -q -n "$DATASET_ZIP" -d /content/data

## Manifest with time groups

In [ ]:
from manifest_utils import build_manifest, summarize_time_groups

df = build_manifest(DATASET_ROOT)
summarize_time_groups(df)

In [ ]:
stats = summarize_time_groups(df)
assert stats['total_images'] == 4390
assert stats['unique_groups'] == 3226
assert stats['multi_image_groups'] == 491
assert stats['pct_images_in_multi_image_groups'] == 37.7
print('time-group statistics match the specification')

## Split and verify

In [ ]:
from split_utils import stratified_group_split, verify_split, save_split

split_df = stratified_group_split(df)
verify_split(split_df)  # raises on leakage, missing class, or wrong total

Image-level proportions land near 70/15/15 rather than exactly on it, because groups differ in size. That is expected and is not corrected for.

In [ ]:
save_split(split_df, '/content/repo/02_Manifests/split_manifest.csv')
df.to_csv('/content/repo/02_Manifests/manifest_full.csv', index=False)
print('saved manifest_full.csv and split_manifest.csv')

Both files are committed to the repository, so training notebooks read them directly and never redraw the split.